# 04 — FastAPI + Docker 데모 서빙

## 목표
파인튜닝된 Qwen2-VL-2B 모델을 REST API로 서빙하고 Docker로 패키징

## 구성
```
vlm_driving/
├── app/
│   └── main.py          ← FastAPI 앱
├── lora_adapter/        ← 파인튜닝된 LoRA 가중치
├── Dockerfile
├── requirements.txt
└── 04_demo_serving.ipynb  ← 이 파일
```

## API 엔드포인트
| 메서드 | 경로 | 설명 |
|--------|------|------|
| GET  | `/health`         | 서버·모델 상태 확인 |
| POST | `/predict`        | 이미지 + 질문 → 답변 |
| POST | `/predict/batch`  | 이미지 한 장 → 7가지 VQA 결과 |

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import subprocess, time, json, requests, threading
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image

print('라이브러리 로드 완료')

## 셀 1: FastAPI 서버 로컬 실행

노트북 안에서 uvicorn을 백그라운드 스레드로 띄웁니다.

In [ ]:
import sys, os
os.chdir(Path('__file__').parent if '__file__' in dir() else Path.cwd())

# vlm_driving/ 폴더에서 실행 (이 노트북 위치 기준)
SERVER_URL = 'http://127.0.0.1:8000'

def _run_server():
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'app.main:app',
         '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning'],
        cwd=str(Path.cwd()),
    )

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# 서버 준비 대기 (모델 로드까지 최대 3분)
print('서버 시작 중... 모델 로드를 기다립니다 (최대 3분)')
for i in range(36):  # 36 × 5초 = 3분
    time.sleep(5)
    try:
        r = requests.get(f'{SERVER_URL}/health', timeout=3)
        if r.status_code == 200 and r.json().get('model_loaded'):
            print(f'\n서버 준비 완료! ({(i+1)*5}초 소요)')
            print(json.dumps(r.json(), ensure_ascii=False, indent=2))
            break
        else:
            print(f'  [{(i+1)*5}s] 모델 로드 중...', end='\r')
    except Exception:
        print(f'  [{(i+1)*5}s] 서버 시작 대기...', end='\r')
else:
    print('\n타임아웃 — 서버 로그를 확인해주세요')

## 셀 2: `/predict` — 단일 질문 테스트

In [ ]:
# 테스트 이미지 선택
import glob
images = sorted(glob.glob('../phase4_carla/data_collection/carla_dataset/images/*.jpg'))
TEST_IMAGE = images[10] if images else None

if not TEST_IMAGE:
    print('이미지를 찾을 수 없습니다.')
else:
    question = '현재 주행 상황에서 즉각적인 위험 요소가 있습니까?'

    with open(TEST_IMAGE, 'rb') as f:
        resp = requests.post(
            f'{SERVER_URL}/predict',
            files={'file': ('image.jpg', f, 'image/jpeg')},
            data={'question': question},
        )

    result = resp.json()

    # 결과 출력
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(Image.open(TEST_IMAGE))
    axes[0].set_title(f'입력 이미지: {Path(TEST_IMAGE).name}', fontsize=11)
    axes[0].axis('off')

    axes[1].axis('off')
    axes[1].set_facecolor('#f8f9fa')
    text = (
        f"[질문]\n{result['question']}\n\n"
        f"[답변]\n{result['answer']}\n\n"
        f"추론 시간: {result['elapsed_sec']}초"
    )
    axes[1].text(0.05, 0.95, text, va='top', ha='left', fontsize=11,
                 wrap=True, transform=axes[1].transAxes,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig('demo_single.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n상태코드: {resp.status_code}')
    print(json.dumps(result, ensure_ascii=False, indent=2))

## 셀 3: `/predict/batch` — 7가지 VQA 한 번에

In [ ]:
TEST_IMAGE_BATCH = images[20] if len(images) > 20 else TEST_IMAGE

with open(TEST_IMAGE_BATCH, 'rb') as f:
    resp = requests.post(
        f'{SERVER_URL}/predict/batch',
        files={'file': ('image.jpg', f, 'image/jpeg')},
    )

result = resp.json()

# 시각화
fig = plt.figure(figsize=(14, 8))

# 왼쪽: 이미지
ax_img = fig.add_axes([0.0, 0.0, 0.38, 1.0])
ax_img.imshow(Image.open(TEST_IMAGE_BATCH))
ax_img.set_title(Path(TEST_IMAGE_BATCH).name, fontsize=10)
ax_img.axis('off')

# 오른쪽: 결과 텍스트
ax_txt = fig.add_axes([0.40, 0.0, 0.60, 1.0])
ax_txt.axis('off')
ax_txt.set_facecolor('#fafafa')

label_map = {
    'scene_description':     '장면 설명',
    'danger_assessment':     '위험 평가',
    'action_recommendation': '행동 추천',
    'pedestrian_check':      '보행자 확인',
    'nearest_object':        '최근접 객체',
    'object_count':          '객체 수',
    'safety_distance':       '안전거리',
}

lines = [f'[{result["elapsed_sec"]}초 소요]\n']
for key, label in label_map.items():
    answer = result['results'].get(key, '-')
    lines.append(f'▶ {label}\n  {answer}\n')

ax_txt.text(0.02, 0.98, '\n'.join(lines), va='top', ha='left', fontsize=9.5,
            transform=ax_txt.transAxes, family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

plt.suptitle('Autonomous Driving VQA — Batch 결과', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('demo_batch.png', dpi=150, bbox_inches='tight')
plt.show()

## 셀 4: API 응답 시간 벤치마크

10회 반복 → 평균/최소/최대 측정

In [ ]:
N = 10
elapsed_list = []

print(f'{N}회 반복 벤치마크...')
for i in range(N):
    img_path = images[i % len(images)]
    with open(img_path, 'rb') as f:
        resp = requests.post(
            f'{SERVER_URL}/predict',
            files={'file': ('image.jpg', f, 'image/jpeg')},
            data={'question': '현재 주행 상황에서 즉각적인 위험 요소가 있습니까?'},
        )
    t = resp.json().get('elapsed_sec', 0)
    elapsed_list.append(t)
    print(f'  [{i+1:2d}/{N}] {t:.3f}초')

import numpy as np
print()
print('=' * 40)
print(f'  평균  : {np.mean(elapsed_list):.3f}초')
print(f'  최소  : {np.min(elapsed_list):.3f}초')
print(f'  최대  : {np.max(elapsed_list):.3f}초')
print(f'  처리량: {1/np.mean(elapsed_list):.2f} 요청/초')
print('=' * 40)

# 시각화
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, N+1), elapsed_list, color='steelblue', alpha=0.8)
ax.axhline(np.mean(elapsed_list), color='red', linestyle='--',
           label=f'평균 {np.mean(elapsed_list):.3f}초')
ax.set_xlabel('요청 번호')
ax.set_ylabel('응답 시간 (초)')
ax.set_title('API 응답 시간 벤치마크 (RTX 4080 SUPER)', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('demo_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## 셀 5: Docker 빌드 & 실행

> Docker Desktop이 설치되어 있어야 합니다.
> GPU 지원은 WSL2 + NVIDIA Container Toolkit 필요.

In [ ]:
# Docker 빌드 (vlm_driving/ 폴더 기준)
# 처음 빌드는 10~20분 소요 (이미지 다운로드 포함)

print('Docker 이미지 빌드 시작...')
print('(처음 빌드는 10~20분 소요)')
print()
print('── 터미널에서 직접 실행하려면 ──')
print('  cd vlm_driving')
print('  docker build -t vlm-driving .')
print()
print('── GPU 있는 경우 실행 ──')
print('  docker run --gpus all -p 8000:8000 vlm-driving')
print()
print('── GPU 없는 경우 (CPU 추론, 느림) ──')
print('  docker run -p 8000:8000 vlm-driving')
print()

# 노트북에서 빌드하려면 아래 주석 해제
# result = subprocess.run(
#     ['docker', 'build', '-t', 'vlm-driving', '.'],
#     cwd=str(Path.cwd()),
#     capture_output=True, text=True
# )
# print(result.stdout[-2000:])  # 마지막 2000자만 출력
# if result.returncode == 0:
#     print('빌드 성공!')
# else:
#     print('빌드 실패:', result.stderr[-500:])

## 셀 6: Swagger UI 확인

FastAPI는 자동으로 API 문서를 생성합니다.

In [ ]:
from IPython.display import IFrame, display, Markdown

display(Markdown(f"""
## Swagger UI (API 문서)

아래 링크를 브라우저에서 열면 API를 직접 테스트할 수 있습니다:

- **Swagger UI**: [{SERVER_URL}/docs]({SERVER_URL}/docs)
- **ReDoc**:       [{SERVER_URL}/redoc]({SERVER_URL}/redoc)
- **Health**:      [{SERVER_URL}/health]({SERVER_URL}/health)

### curl 예시
```bash
# 단일 질문
curl -X POST {SERVER_URL}/predict \\
  -F 'file=@your_image.jpg' \\
  -F 'question=현재 주행 상황에서 즉각적인 위험 요소가 있습니까?'

# 7가지 VQA 일괄
curl -X POST {SERVER_URL}/predict/batch \\
  -F 'file=@your_image.jpg'
```
"""))

## 셀 7: 프로젝트 전체 완성 요약

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════╗
║      Autonomous CV Pipeline — 프로젝트 완성 요약             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Phase 1 — CV 기초                                           ║
║    Detection  : YOLOv8  mAP@50=0.99                         ║
║    Segmentation: SAM2   mIoU=0.694                           ║
║    Depth      : DepthAnythingV2  RMSE=0.16m                  ║
║                                                              ║
║  Phase 2 — Tracking + Pose                                   ║
║    ByteTrack  : MOTA=0.9412 (직접 구현)                      ║
║    Pose       : YOLOv8-pose + OKS 파이프라인                  ║
║                                                              ║
║  Phase 3 — BEV                                               ║
║    IPM        : 핀홀 카메라 + Homography 직접 구현            ║
║                                                              ║
║  Phase 4 — CARLA 통합 + 파인튜닝                             ║
║    YOLOv8     : mAP 0.43 → 0.68  (+59%)                     ║
║    SegFormer  : mIoU 0.107 → 0.586  (+448%)                  ║
║    Depth      : RMSE 5.44 → 2.71m  (-50%)                    ║
║                                                              ║
║  VLM — Qwen2-VL-2B QLoRA                                     ║
║    학습 데이터: 2,243개 (CARLA 자동 생성)                     ║
║    BLEU       : 0.004 → 0.564  (+141배)                      ║
║    ROUGE-L    : 0.027 → 0.759  (+28배)                       ║
║    거리 MAE   : 5.36m  (5m 이내 68%)                         ║
║                                                              ║
║  배포 — FastAPI + Docker                                      ║
║    /predict       : 이미지 + 질문 → 답변                      ║
║    /predict/batch : 7가지 VQA 일괄 처리                       ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)